<a href="https://colab.research.google.com/github/md-vasim/LLMs/blob/main/Generative-AI-With-LLM-DLAI/lab_2_instruction_finetuning_dialogue_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --upgrade pip
%pip install --disable-pip-version-check \
    torch==1.13.1 \
    torchdata==0.5.1 --quiet

%pip install \
    transformers==4.27.2 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    loralib==0.1.1 \
    peft==0.3.0 --quiet

%pip install -U datasets fsspec --quiet 

In [ ]:
from datasets import load_dataset
import torch
import time
import evaluate
import pandas as pd
import numpy as np
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    GenerationConfig,
    TrainingArguments,
    Trainer
)

### Dataset

In [ ]:
hf_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(hf_dataset_name)
dataset

In [ ]:
model_name = 'google/flan-t5-base'
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def print_number_of_trainable_model_params(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: "

In [ ]:
print(print_number_of_trainable_model_params(original_model))

### Zero Shot Inferencing

In [ ]:
index = 200

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

inputs = tokenizer(prompt, return_tensors='pt')
output = tokenizer.decode(
    original_model.generate(
        inputs['input_ids'],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)
dash_line = '-'.join('' for x in range(100))
print(dash_line)
print("Example")
print(dash_line)
print(f"INPUT PROMPT:\n{prompt}")
print(dataset['test'][index]['dialogue'])
print(f"BASELINE HUMAN SUMMARY:\n{summary}")
print(dash_line)
print(f"\nMODEL GENERATION - ZERO SHOT: \n{output}\n")

In [ ]:
def tokenize_function(example):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [
        start_prompt+dialogue+end_prompt for dialogue in example['dialogue']
    ]
    example['input_ids'] = tokenizer(prompt, padding='max_length', truncation=True, return_tensors='pt').input_ids
    example['labels'] = tokenizer(example['summary'], padding='max_length', truncation=True, return_tensors='pt').input_ids

    return example

# The dataset actually contains 3 diff splits: train, validation, test
# The tokenize function code is handling all data across all splits in batches
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(['id', 'dialogue', 'summary'])
tokenized_dataset

In [ ]:
tokenized_dataset = tokenized_dataset.filter(lambda example, index: index % 100 == 0, with_indices=True)
tokenized_dataset

In [ ]:
print(f'Shapes of the datasets: ')
print(f'train: {tokenized_dataset["train"].shape}')
print(f'validation: {tokenized_dataset["validation"].shape}')
print(f'test: {tokenized_dataset["test"].shape}')


### Fine tuning the model with the processed datasets


In [ ]:
output_dir = f'./models'

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=1,
    max_steps=1
)

trainer = Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation']
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model('./models')

### Instruc model

In [ ]:
instruct_model = AutoModelForSeq2SeqLM.from_pretrained('./models', torch_dtype=torch.bfloat16).to('cuda')

In [ ]:
index = 200

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
original_model_output = tokenizer.decode(
    original_model.generate(
        inputs['input_ids'],
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
)

instruct_model_output = tokenizer.decode(
    instruct_model.generate(
        inputs['input_ids'],
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print("Example")
print(dash_line)
print(f"INPUT PROMPT:\n{prompt}")
print(dataset['test'][index]['dialogue'])
print(f"BASELINE HUMAN SUMMARY:\n{summary}")
print(dash_line)
print(f"\nMODEL GENERATION - Original model: \n{original_model_output}\n")
print(dash_line)
print(f"\nMODEL GENERATION - Instruct model: \n{instruct_model_output}\n")

### Evaluation

In [ ]:
rouge = evaluate.load('rouge')

In [ ]:
dialogues = dataset['test'][0:10]['dialogue']
human_baseline_summaries = dataset['test'][0:10]['summary']

original_model_summaries = []
instruct_model_summaries = []

for _, dialogue in enumerate(dialogues):
  prompt = f"""
  Summarize the following conversation.

  {dialogue}

  Summary:
  """

  input_ids = tokenizer(prompt, return_tensors='pt').to('cuda').input_ids
  original_model_text_output = tokenizer.decode(
    original_model.generate(
        input_ids,
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
  )
  original_model_summaries.append(original_model_text_output)

  instruct_model_text_output = tokenizer.decode(
      instruct_model.generate(
          input_ids,
          generation_config=GenerationConfig(
              max_new_tokens=200, num_beams=1
          )
      )[0],
      skip_special_tokens=True
  )
  instruct_model_summaries.append(instruct_model_text_output)

zipped_summaries = list(zip(human_baseline_summaries, original_model_summaries, instruct_model_summaries))
df = pd.DataFrame(zipped_summaries, columns=['human_baseline_summary', 'original_model_summary', 'instruct_model_summary'])
df


In [ ]:
original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

print(f'Original model results:')
print(original_model_results)
print(dash_line)
print(f'Instruct model results:')
print(instruct_model_results)

### PEFT

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32, # Rank,
    lora_alpha=32,
    target_modules=['q', 'v'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

In [ ]:
peft_model = get_peft_model(original_model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
print(print_number_of_trainable_model_params(peft_model))

In [ ]:
output_dir = f'./model-peft'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=1
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_dataset['train'],
)

In [ ]:
peft_trainer.train()

In [ ]:
peft_trainer.model.save_pretrained('./model-peft')
tokenizer.save_pretrained('./model-peft')

In [ ]:
from peft  import PeftModel, PeftConfig

peft_model_base = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base', torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

peft_model = PeftModel.from_pretrained(
    peft_model_base, './model-peft', torch_dtype=torch.bfloat16, is_trainable=False
    ).to('cuda')


In [ ]:
print(print_number_of_trainable_model_params(peft_model))

### Evaluation

In [ ]:
index = 200

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

input_ids = tokenizer(prompt, return_tensors='pt').to('cuda').input_ids

original_model_output = tokenizer.decode(
    original_model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
)

instruct_model_output = tokenizer.decode(
    instruct_model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
)

peft_model_output = tokenizer.decode(
    peft_model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print("Example")
print(dash_line)
print(f"INPUT PROMPT:\n{prompt}")
print(dataset['test'][index]['dialogue'])
print(f"BASELINE HUMAN SUMMARY:\n{summary}")
print(dash_line)
print(f"\nMODEL GENERATION - Original model: \n{original_model_output}\n")
print(dash_line)
print(f"\nMODEL GENERATION - Instruct model: \n{instruct_model_output}\n")
print(dash_line)
print(f"\nMODEL GENERATION - Peft model: \n{peft_model_output}\n")

In [ ]:
dialogues = dataset['test'][0:10]['dialogue']
human_baseline_summaries = dataset['test'][0:10]['summary']

original_model_summaries = []
instruct_model_summaries = []
peft_model_summaries = []

for _, dialogue in enumerate(dialogues):
  prompt = f"""
  Summarize the following conversation.

  {dialogue}

  Summary:
  """

  input_ids = tokenizer(prompt, return_tensors='pt').to('cuda').input_ids
  original_model_text_output = tokenizer.decode(
    original_model.generate(
        input_ids,
        generation_config=GenerationConfig(
            max_new_tokens=200, num_beams=1
        )
    )[0],
    skip_special_tokens=True
  )

  instruct_model_text_output = tokenizer.decode(
      instruct_model.generate(
          input_ids,
          generation_config=GenerationConfig(
              max_new_tokens=200, num_beams=1
          )
      )[0],
      skip_special_tokens=True
  )

  peft_model_text_output = tokenizer.decode(
      peft_model.generate(
          input_ids=input_ids,
          generation_config=GenerationConfig(
              max_new_tokens=200, num_beams=1
          )
      )[0],
      skip_special_tokens=True
  )

  original_model_summaries.append(original_model_text_output)
  instruct_model_summaries.append(instruct_model_text_output)
  peft_model_summaries.append(peft_model_text_output)

zipped_summaries = list(
    zip(human_baseline_summaries, original_model_summaries, instruct_model_summaries, peft_model_summaries
        ))
df = pd.DataFrame(zipped_summaries,
                  columns=[
                      'human_baseline_summary', 'original_model_summary',
                      'instruct_model_summary', 'peft_model_summaries'])
df

In [ ]:
rouge = evaluate.load('rouge')

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries[0:len(instruct_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries[0:len(peft_model_summaries)],
    use_aggregator=True,
    use_stemmer=True
)

print(f'Original model results:')
print(original_model_results)
print(dash_line)
print(f'Instruct model results:')
print(instruct_model_results)
print(dash_line)
print(f'Peft model results:')
print(peft_model_results)